In [20]:
# Starting the omlx server
!/opt/homebrew/bin/omlx start

oMLX server running on port 8000


In [21]:
#import statements and keys


import os
from pathlib import Path
from dotenv import load_dotenv
import os
import yaml
import logging
import re
from datetime import datetime
import ast
import yaml
import time
from openai import OpenAI

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
env_path = Path.cwd().parent / "environment.env"

# Load the .env file
if load_dotenv(dotenv_path=env_path):
    print("Environment variables loaded successfully!")
else:
    print("Error: Could not find or load the .env file.")

OmlxAPI = os.getenv('OmlxAPI')
baseurl = os.getenv('baseurl')


Environment variables loaded successfully!


In [22]:
def llm_response_qwen(prompt: str, ) -> str:
    """Generic function to get a response from the Groq Qwen LLM."""
    try:
        client = OpenAI(
            base_url=baseurl,  # Your oMLX local address
            api_key=OmlxAPI          # Your oMLX API Key
        )
        response = client.chat.completions.create(
            model="mlx-community/gemma-4-12B-it-OptiQ-4bit",           # Replace with your loaded model ID
            messages=[{"role": "user", "content": prompt}]
        )
             
        content = response.choices[0].message.content
        return content
    except Exception as e:
        logging.error(f"An error occurred while communicating with the Groq API: {e}")
        return ""

In [26]:
def yamlcheck(content):
    try:
        raw_response = content.strip()
    except:
        None

    # Remove markdown code fences if Qwen accidentally includes them
    if raw_response.startswith("```"):
        raw_response = (raw_response.replace("```python", "").replace("```", "").strip()
        )

    try:
        # Safely convert the string representation of a list into a real Python list
        content = ast.literal_eval(raw_response)
    except Exception as e:
        print(f"Error parsing list: {e}")
        # Fallback: just split by lines if it completely fails
        content = [line.strip() for line in raw_response.split("\n") if line.strip()]
        
    return content

In [23]:
jobdescription = input('Enter the job description')

In [67]:
#Load the content files

path = '/Users/karthik/Documents/Github/Colab/Resume.yaml'
with open(path, 'r') as f:
    resume = yaml.safe_load(f)
# a = list(resume['Professional Experience'].keys())

#Load Destination Files
path = '/Users/karthik/Documents/Github/Colab/output copy.yaml'
with open(path, 'r') as f:
    test = yaml.safe_load(f)



#Create loop to get tailored initial responses with checking all the experiences are available

if (len(resume["work_experience"]) ==len(test['work_experience'])):
    for i in range(len(resume["work_experience"])):
    # for i in range(1):
        experience = resume["work_experience"][i]["achievements"]
        prompt1 = f"""[Task] Rewrite exactly 7 resume bullet points to align with the Job Description.

        [Constraints]
        - Truthfulness: Use ONLY metrics, tools, and software explicitly in the Resume. Never add/change tools or numbers.
        - Format: Return ONLY a raw, valid Python list of 7 strings on ONE continuous line. No markdown blocks (```), no \n, no indents. Start with [ and end with ].
        - Content: Blend problem, solution, and impact seamlessly. No labels like "Problem:" or "Impact:".

        [Example]
        Resume: - Built SQL tracking systems for supplier monitoring, cutting reporting time by 50%.
        Job: Optimize vendor workflows and leverage data analytics.
        Output: ["Optimized vendor workflows by building SQL tracking systems to automate supplier monitoring, cutting reporting time by 50%."]

        [Data]
        Resume: {experience}
        Job: {jobdescription}

        [Guardrail] Ensure exactly 7 elements are in the list. Zero invented skills/tools."""

    
        tailoredresponse = llm_response_qwen(prompt1)
        cleanedresponse = yamlcheck(tailoredresponse)

        test['work_experience'][i]['achievements'] = cleanedresponse

    # Now saving to YAML will be perfectly clean and structured
    with open("output.yaml", "w", encoding="utf-8") as f:
        yaml.dump(test, f, sort_keys=False, default_flow_style=False)

    print("inital tailored resume collected")
else:
    print("There is not same number of experiences in source and destination")

2026-06-30 19:23:39,606 - INFO - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-30 19:23:50,987 - INFO - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-30 19:24:05,606 - INFO - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-30 19:24:21,073 - INFO - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"


inital tailored resume collected


In [ ]:
path = '/Users/karthik/Documents/Github/Colab/output copy.yaml'
with open(path, 'r') as f:
    test = yaml.safe_load(f)

output = [{"name": "Core Competencies", "keywords": [ "Strategic Procurement", "Supplier Negotiation", "Procurement Analytics"]}]


skillsprompt = f""" Rewrite the skills section in a crisp one or two word to suit the below tailored resume and job description maintain same yaml format can modify or create new skill sections and rate the final skills plus reume for ats compatablity to the job description.
- Format: Return ONLY a raw, valid YAML format. No markdown blocks (```), no \n, no indents. Start with [ and end with ].

skills: {test['skills']}
Job: {jobdescription}
resume: {test['work_experience']}

Example:
OUTPUT : {output}

NO PREAMBLE
"""


tailoredresponse = llm_response_qwen(skillsprompt)
cleanedresponse = yamlcheck(tailoredresponse)
test['skills'] = cleanedresponse[0]['skills']
with open("output.yaml", "w", encoding="utf-8") as f:
        yaml.dump(test, f, sort_keys=False, default_flow_style=False)

2026-06-30 20:43:55,611 - INFO - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"


In [ ]:
path = '/Users/karthik/Documents/Github/Colab/output copy.yaml'
with open(path, 'r') as f:
    test = yaml.safe_load(f)
output = {"summary":"Strategic Procurement and Sourcing Professional with over 7 years of experience in "}

summaryprompt = f"""rewrite the summary section to suit the job description and resume
- Format: Return ONLY a raw, valid YAML format. No markdown blocks (```), no \n, no indents. Start with [ and end with ].

Output : {output}

summary: {test['summary']}
Job: {jobdescription}
resume: {test['work_experience']}



NO PREAMBLE
"""

tailoredresponse = llm_response_qwen(summaryprompt)
cleanedresponse = yamlcheck(tailoredresponse)
test['summary'] = cleanedresponse['summary']
with open("output.yaml", "w", encoding="utf-8") as f:
        yaml.dump(test, f, sort_keys=False, default_flow_style=False)


2026-06-30 20:54:46,984 - INFO - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"


In [84]:
path = '/Users/karthik/Documents/Github/Colab/output copy.yaml'
with open(path, 'r') as f:
    test = yaml.safe_load(f)
    
output = {"project_name": "Procurement Analytics Dashboard",
    "description": "Developed a Power BI dashboard to track procurement KPIs, supplier performance, and category spend, providing actionable insights for informed purchasing decisions.",
    "keywords" : ["Power BI", "Procurement", "Supply Chain", "KPIs"]}

projectprompt = f"""Select/ Rewrite 4 projects from the below list to suit the job description

- Format: Return ONLY a raw, valid YAML format. No markdown blocks (```), no \n, no indents. Start with [ and end with ].

Output : {output}

projects: {resume['projects']}
Job: {jobdescription}

NO PREAMBLE

"""

tailoredresponse = llm_response_qwen(projectprompt)
cleanedresponse = yamlcheck(tailoredresponse)

test['projects']= cleanedresponse
with open("output.yaml", "w", encoding="utf-8") as f:
        yaml.dump(test, f, sort_keys=False, default_flow_style=False)

2026-06-30 21:05:48,818 - INFO - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"


In [4]:
import yaml
path = '/Users/karthik/Documents/Github/Colab/output copy.yaml'
with open(path, 'r') as f:
    test = yaml.safe_load(f)

In [9]:
prompt = f""" match between the resume and the job give me a score out of 10
    Resume: {test}
    Job: {jobdescription}
"""

In [10]:
prompt

" match between the resume and the job give me a score out of 10\n    Resume: {'summary': 'Strategic Procurement and Supply Chain Analyst with 8+ years of experience leading sourcing, supplier management, procurement operations, and cost optimization initiatives across global manufacturing and industrial environments. Proven expertise in strategic sourcing, vendor negotiations, procurement analytics, ERP systems, and supplier performance management using Power BI, SQL, and Excel. Experienced managing end-to-end purchasing operations including purchase orders, shipment tracking, lead-time monitoring, contract management, and supplier risk mitigation. Adept at collaborating cross-functionally with operations, engineering, finance, and sales teams to improve supply continuity, operational efficiency, and procurement performance.', 'skills': [{'name': 'Procurement', 'keywords': ['Strategic Sourcing', 'RFx', 'Vendor Management', 'PO Management', 'Category Management']}, {'name': 'Supplier M

prompt

In [11]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8000/v1",  # Your oMLX local address
    api_key="1234"          # Your oMLX API Key
)

response = client.chat.completions.create(
    model="mlx-community/gemma-4-12B-it-OptiQ-4bit",           # Replace with your loaded model ID
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

### **Match Score: 9.2/10**

#### **Analysis Overview**
The candidate is an exceptionally strong match for the **Senior Procurement Advisor** role at BC Hydro. The resume demonstrates a clear progression of 8+ years in procurement, moving from analyst roles to high-level strategic procurement and data-driven leadership. The candidate’s technical proficiency in analytics (Power BI, SQL, Python) combined with deep "hard" procurement skills (RFx, Contract Negotiation, Risk Mitigation) aligns perfectly with the requirements for managing "medium and high complexity contracts."

---

#### **Strengths (Why the score is high):**
*   **Experience Level:** The job requires a minimum of 8 years of procurement experience; the candidate explicitly lists 8+ years with a history of "progressively senior roles."
*   **Hard Skills Alignment:** The candidate has mastered every core task listed in the "What you'll do" section:
    *   **RFx Management:** Explicit experience with RFT, RFP, RFQ, RFI, and R

In [33]:
type(response)

openai.types.chat.chat_completion.ChatCompletion

In [38]:
# 1. Extract the raw string from the object
raw_content = response.choices[0].message.content

print(raw_content)

["Spearheaded strategic sourcing initiatives that slashed procurement costs while diversifying the supplier base to lower organizational risk and optimize sourcing methods.", "Managed the end-to-end Procure-to-Pay (P2P) lifecycle to achieve a 98% on-time delivery rate and ensure seamless invoice reconciliation for high-value services.", "Designed interactive Power BI dashboards to turn procurement data into actionable insights, enabling leadership to identify project requirements and financial considerations.", "Developed procurement strategies to reduce stockouts and maintain 98% production uptime by re-engineering purchase planning for global suppliers with long lead times.", "Boosted vendor accountability and lead-time reliability through a high-engagement relationship model, resulting in a 30% improvement in on-time deliveries.", "Led cross-functional 'Continuous Improvement' sessions to identify and roll out business process improvements that enhanced operational efficiency and sc

In [39]:
raw_content

'["Spearheaded strategic sourcing initiatives that slashed procurement costs while diversifying the supplier base to lower organizational risk and optimize sourcing methods.", "Managed the end-to-end Procure-to-Pay (P2P) lifecycle to achieve a 98% on-time delivery rate and ensure seamless invoice reconciliation for high-value services.", "Designed interactive Power BI dashboards to turn procurement data into actionable insights, enabling leadership to identify project requirements and financial considerations.", "Developed procurement strategies to reduce stockouts and maintain 98% production uptime by re-engineering purchase planning for global suppliers with long lead times.", "Boosted vendor accountability and lead-time reliability through a high-engagement relationship model, resulting in a 30% improvement in on-time deliveries.", "Led cross-functional \'Continuous Improvement\' sessions to identify and roll out business process improvements that enhanced operational efficiency and

In [34]:
content = ""
for chunk in response:
    print(chunk)
    break

    # c = chunk.choices[0].delta.content if chunk.choices[0].delta.content else ""
    # content += c

('id', 'chatcmpl-6b2bb1aa')


In [10]:
path = '/Users/karthik/Documents/Github/Colab/Resume.yaml'
with open(path, 'r') as f:
    resume = yaml.safe_load(f)

In [1]:
resume

NameError: name 'resume' is not defined

In [26]:
answer = ["Automated supplier performance monitoring and provided early warnings for dependency risks by building custom SQL-based tracking systems, cutting manual reporting time by 50%.", "Developed procurement and contract documents by designing interactive Power BI dashboards that transformed messy procurement data into actionable insights to speed up sourcing workflows.", "Maintained a 'single source of truth' for end-to-end supply chain visibility and data accuracy within Sage ERP to support leadership decision-making.", "Streamlined data mining and reporting processes by engineering complex queries to transform raw numbers into monthly performance reports.", "Reduced stockouts and achieved 98% production uptime by re-engineering purchase planning for global suppliers with long lead times.", "Spearheaded strategic sourcing initiatives that slashed procurement costs while diversifying the supplier base to lower organizational risk.", "Improved vendor accountability and lead-time reliability through a high-engagement relationship model, resulting in a 30% improvement in on-time deliveries."]

In [27]:
answer

['Automated supplier performance monitoring and provided early warnings for dependency risks by building custom SQL-based tracking systems, cutting manual reporting time by 50%.',
 'Developed procurement and contract documents by designing interactive Power BI dashboards that transformed messy procurement data into actionable insights to speed up sourcing workflows.',
 "Maintained a 'single source of truth' for end-to-end supply chain visibility and data accuracy within Sage ERP to support leadership decision-making.",
 'Streamlined data mining and reporting processes by engineering complex queries to transform raw numbers into monthly performance reports.',
 'Reduced stockouts and achieved 98% production uptime by re-engineering purchase planning for global suppliers with long lead times.',
 'Spearheaded strategic sourcing initiatives that slashed procurement costs while diversifying the supplier base to lower organizational risk.',
 'Improved vendor accountability and lead-time relia

In [2]:
!pkill -f "omlx serve"
